# World Cup 2022 Data Prep

## Objective
Prepare clean, analysis-ready datasets for tournament simulation and modeling.

## Outputs
- `data/processed/worldcup_2022_fixtures.csv`
- `data/processed/worldcup_2022_teams.csv`
- `data/processed/international_training_matches.csv`

In [ ]:
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
if PROJECT_ROOT.name != 'Project-3-WorldCup-2022-Predictions':
    PROJECT_ROOT = Path.cwd().resolve()

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'

for p in [RAW_DIR, PROCESSED_DIR, OUTPUTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUTS_DIR

## Data Assumptions
- Team names are normalized to a single canonical format.
- Fixture list includes all 64 official matches (group + knockout, including third-place playoff).
- Historical training schema remains fixed: `date`, `home_team`, `away_team`, `home_goals`, `away_goals`, `tournament`, `neutral`.

In [ ]:
wc_groups = {
    'A': ['Qatar', 'Ecuador', 'Senegal', 'Netherlands'],
    'B': ['England', 'Iran', 'USA', 'Wales'],
    'C': ['Argentina', 'Saudi Arabia', 'Mexico', 'Poland'],
    'D': ['France', 'Australia', 'Denmark', 'Tunisia'],
    'E': ['Spain', 'Costa Rica', 'Germany', 'Japan'],
    'F': ['Belgium', 'Canada', 'Morocco', 'Croatia'],
    'G': ['Brazil', 'Serbia', 'Switzerland', 'Cameroon'],
    'H': ['Portugal', 'Ghana', 'Uruguay', 'South Korea']
}
wc_groups

In [ ]:
fixture_rows = [
    (1, '2022-11-20', 'Group', 'A', 'Qatar', 'Ecuador', 0, 2),
    (2, '2022-11-21', 'Group', 'A', 'Senegal', 'Netherlands', 0, 2),
    (3, '2022-11-21', 'Group', 'B', 'England', 'Iran', 6, 2),
    (4, '2022-11-21', 'Group', 'B', 'USA', 'Wales', 1, 1),
    (5, '2022-11-22', 'Group', 'C', 'Argentina', 'Saudi Arabia', 1, 2),
    (6, '2022-11-22', 'Group', 'C', 'Mexico', 'Poland', 0, 0),
    (7, '2022-11-22', 'Group', 'D', 'Denmark', 'Tunisia', 0, 0),
    (8, '2022-11-22', 'Group', 'D', 'France', 'Australia', 4, 1),
    (9, '2022-11-23', 'Group', 'E', 'Germany', 'Japan', 1, 2),
    (10, '2022-11-23', 'Group', 'E', 'Spain', 'Costa Rica', 7, 0),
    (11, '2022-11-23', 'Group', 'F', 'Morocco', 'Croatia', 0, 0),
    (12, '2022-11-23', 'Group', 'F', 'Belgium', 'Canada', 1, 0),
    (13, '2022-11-24', 'Group', 'G', 'Switzerland', 'Cameroon', 1, 0),
    (14, '2022-11-24', 'Group', 'G', 'Brazil', 'Serbia', 2, 0),
    (15, '2022-11-24', 'Group', 'H', 'Uruguay', 'South Korea', 0, 0),
    (16, '2022-11-24', 'Group', 'H', 'Portugal', 'Ghana', 3, 2),
    (17, '2022-11-25', 'Group', 'A', 'Qatar', 'Senegal', 1, 3),
    (18, '2022-11-25', 'Group', 'A', 'Netherlands', 'Ecuador', 1, 1),
    (19, '2022-11-25', 'Group', 'B', 'Wales', 'Iran', 0, 2),
    (20, '2022-11-25', 'Group', 'B', 'England', 'USA', 0, 0),
    (21, '2022-11-26', 'Group', 'D', 'Tunisia', 'Australia', 0, 1),
    (22, '2022-11-26', 'Group', 'D', 'France', 'Denmark', 2, 1),
    (23, '2022-11-26', 'Group', 'C', 'Poland', 'Saudi Arabia', 2, 0),
    (24, '2022-11-26', 'Group', 'C', 'Argentina', 'Mexico', 2, 0),
    (25, '2022-11-27', 'Group', 'E', 'Japan', 'Costa Rica', 0, 1),
    (26, '2022-11-27', 'Group', 'E', 'Spain', 'Germany', 1, 1),
    (27, '2022-11-27', 'Group', 'F', 'Belgium', 'Morocco', 0, 2),
    (28, '2022-11-27', 'Group', 'F', 'Croatia', 'Canada', 4, 1),
    (29, '2022-11-28', 'Group', 'G', 'Cameroon', 'Serbia', 3, 3),
    (30, '2022-11-28', 'Group', 'G', 'Brazil', 'Switzerland', 1, 0),
    (31, '2022-11-28', 'Group', 'H', 'South Korea', 'Ghana', 2, 3),
    (32, '2022-11-28', 'Group', 'H', 'Portugal', 'Uruguay', 2, 0),
    (33, '2022-11-29', 'Group', 'A', 'Ecuador', 'Senegal', 1, 2),
    (34, '2022-11-29', 'Group', 'A', 'Netherlands', 'Qatar', 2, 0),
    (35, '2022-11-29', 'Group', 'B', 'Wales', 'England', 0, 3),
    (36, '2022-11-29', 'Group', 'B', 'Iran', 'USA', 0, 1),
    (37, '2022-11-30', 'Group', 'D', 'Australia', 'Denmark', 1, 0),
    (38, '2022-11-30', 'Group', 'D', 'Tunisia', 'France', 1, 0),
    (39, '2022-11-30', 'Group', 'C', 'Poland', 'Argentina', 0, 2),
    (40, '2022-11-30', 'Group', 'C', 'Saudi Arabia', 'Mexico', 1, 2),
    (41, '2022-12-01', 'Group', 'F', 'Croatia', 'Belgium', 0, 0),
    (42, '2022-12-01', 'Group', 'F', 'Canada', 'Morocco', 1, 2),
    (43, '2022-12-01', 'Group', 'E', 'Japan', 'Spain', 2, 1),
    (44, '2022-12-01', 'Group', 'E', 'Costa Rica', 'Germany', 2, 4),
    (45, '2022-12-02', 'Group', 'G', 'Serbia', 'Switzerland', 2, 3),
    (46, '2022-12-02', 'Group', 'G', 'Cameroon', 'Brazil', 1, 0),
    (47, '2022-12-02', 'Group', 'H', 'South Korea', 'Portugal', 2, 1),
    (48, '2022-12-02', 'Group', 'H', 'Ghana', 'Uruguay', 0, 2),
    (49, '2022-12-03', 'R16', np.nan, 'Netherlands', 'USA', 3, 1),
    (50, '2022-12-03', 'R16', np.nan, 'Argentina', 'Australia', 2, 1),
    (51, '2022-12-04', 'R16', np.nan, 'France', 'Poland', 3, 1),
    (52, '2022-12-04', 'R16', np.nan, 'England', 'Senegal', 3, 0),
    (53, '2022-12-05', 'R16', np.nan, 'Japan', 'Croatia', 1, 1),
    (54, '2022-12-05', 'R16', np.nan, 'Brazil', 'South Korea', 4, 1),
    (55, '2022-12-06', 'R16', np.nan, 'Morocco', 'Spain', 0, 0),
    (56, '2022-12-06', 'R16', np.nan, 'Portugal', 'Switzerland', 6, 1),
    (57, '2022-12-09', 'QF', np.nan, 'Croatia', 'Brazil', 1, 1),
    (58, '2022-12-09', 'QF', np.nan, 'Netherlands', 'Argentina', 2, 2),
    (59, '2022-12-10', 'QF', np.nan, 'Morocco', 'Portugal', 1, 0),
    (60, '2022-12-10', 'QF', np.nan, 'England', 'France', 1, 2),
    (61, '2022-12-13', 'SF', np.nan, 'Argentina', 'Croatia', 3, 0),
    (62, '2022-12-14', 'SF', np.nan, 'France', 'Morocco', 2, 0),
    (63, '2022-12-17', 'Third Place', np.nan, 'Croatia', 'Morocco', 2, 1),
    (64, '2022-12-18', 'Final', np.nan, 'Argentina', 'France', 3, 3)
]

fixtures = pd.DataFrame(
    fixture_rows,
    columns=[
        'match_no', 'date', 'stage', 'group', 'home_team', 'away_team',
        'actual_home_goals', 'actual_away_goals'
    ]
)
fixtures.head()

In [ ]:
stage_order = ['Group', 'R16', 'QF', 'SF', 'Third Place', 'Final']

name_map = {
    'Korea Republic': 'South Korea',
    'USA': 'USA'
}

for col in ['home_team', 'away_team']:
    fixtures[col] = fixtures[col].replace(name_map).str.strip()

fixtures['date'] = pd.to_datetime(fixtures['date'])
fixtures['stage'] = pd.Categorical(fixtures['stage'], categories=stage_order, ordered=True)
fixtures = fixtures.sort_values(['date', 'match_no']).reset_index(drop=True)
fixtures.tail()

In [ ]:
assert len(fixtures) == 64, f'Expected 64 matches, found {len(fixtures)}'
assert (fixtures['stage'] == 'Group').sum() == 48, 'Group-stage match count must be 48'
assert (fixtures['stage'] != 'Group').sum() == 16, 'Knockout-stage match count must be 16'
print('Fixture integrity checks passed.')

In [ ]:
confed_map = {
    'Qatar': 'AFC', 'Ecuador': 'CONMEBOL', 'Senegal': 'CAF', 'Netherlands': 'UEFA',
    'England': 'UEFA', 'Iran': 'AFC', 'USA': 'CONCACAF', 'Wales': 'UEFA',
    'Argentina': 'CONMEBOL', 'Saudi Arabia': 'AFC', 'Mexico': 'CONCACAF', 'Poland': 'UEFA',
    'France': 'UEFA', 'Australia': 'AFC', 'Denmark': 'UEFA', 'Tunisia': 'CAF',
    'Spain': 'UEFA', 'Costa Rica': 'CONCACAF', 'Germany': 'UEFA', 'Japan': 'AFC',
    'Belgium': 'UEFA', 'Canada': 'CONCACAF', 'Morocco': 'CAF', 'Croatia': 'UEFA',
    'Brazil': 'CONMEBOL', 'Serbia': 'UEFA', 'Switzerland': 'UEFA', 'Cameroon': 'CAF',
    'Portugal': 'UEFA', 'Ghana': 'CAF', 'Uruguay': 'CONMEBOL', 'South Korea': 'AFC'
}

team_rows = []
for group_name, teams in wc_groups.items():
    for team in teams:
        team_rows.append({
            'team': team,
            'group': group_name,
            'confederation': confed_map.get(team),
            'is_host': team == 'Qatar'
        })

teams_master = pd.DataFrame(team_rows).sort_values(['group', 'team']).reset_index(drop=True)
teams_master.head()

In [ ]:
expected_training_cols = [
    'date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'tournament', 'neutral'
]

def try_load_historical_source(raw_dir: Path):
    for csv_path in sorted(raw_dir.glob('*.csv')):
        candidate = pd.read_csv(csv_path)
        if all(col in candidate.columns for col in expected_training_cols):
            print(f'Using historical source: {csv_path.name}')
            return candidate[expected_training_cols].copy()
    return None

def make_synthetic_historical_data(teams_df: pd.DataFrame, n_matches: int = 1400, seed: int = 42):
    rng = np.random.default_rng(seed)
    teams = teams_df['team'].tolist()
    ratings = {team: rng.normal(0, 0.35) for team in teams}

    rows = []
    date_range = pd.date_range('2017-01-01', '2022-11-10', freq='D')
    for _ in range(n_matches):
        home, away = rng.choice(teams, size=2, replace=False)
        neutral = bool(rng.random() < 0.35)

        home_edge = 0.25 if not neutral else 0.0
        lambda_home = np.clip(np.exp(0.4 + ratings[home] - ratings[away] + home_edge), 0.2, 3.5)
        lambda_away = np.clip(np.exp(0.35 + ratings[away] - ratings[home]), 0.2, 3.2)

        rows.append({
            'date': rng.choice(date_range),
            'home_team': home,
            'away_team': away,
            'home_goals': int(rng.poisson(lambda_home)),
            'away_goals': int(rng.poisson(lambda_away)),
            'tournament': 'International Friendly',
            'neutral': neutral
        })

    return pd.DataFrame(rows).sort_values('date').reset_index(drop=True)

historical = try_load_historical_source(RAW_DIR)
if historical is None:
    print('No valid historical CSV found in data/raw. Generating deterministic synthetic fallback dataset.')
    historical = make_synthetic_historical_data(teams_master)

historical.head()

In [ ]:
historical['date'] = pd.to_datetime(historical['date'])
for col in ['home_team', 'away_team', 'tournament']:
    historical[col] = historical[col].astype(str).str.strip()

historical['neutral'] = historical['neutral'].astype(bool)
historical['goal_diff'] = historical['home_goals'] - historical['away_goals']
historical['result'] = np.select(
    [historical['goal_diff'] > 0, historical['goal_diff'] < 0],
    ['H', 'A'],
    default='D'
)
historical['year'] = historical['date'].dt.year
historical['month'] = historical['date'].dt.month

historical = historical.sort_values('date').reset_index(drop=True)
historical.head()

In [ ]:
fixtures.to_csv(PROCESSED_DIR / 'worldcup_2022_fixtures.csv', index=False)
teams_master.to_csv(PROCESSED_DIR / 'worldcup_2022_teams.csv', index=False)
historical.to_csv(PROCESSED_DIR / 'international_training_matches.csv', index=False)

print('Saved processed outputs to:')
print('-', PROCESSED_DIR / 'worldcup_2022_fixtures.csv')
print('-', PROCESSED_DIR / 'worldcup_2022_teams.csv')
print('-', PROCESSED_DIR / 'international_training_matches.csv')

In [ ]:
display(fixtures.head())
display(fixtures.groupby('stage').size().rename('matches_by_stage'))
display(fixtures[fixtures['stage'] == 'Group'].groupby('group').size().rename('group_stage_match_counts'))
display(teams_master.head())
display(historical.head())

## Handoff To Notebook 2
Notebook 2 should load the three processed CSV files and build only pre-match features for model training and tournament simulation.